<a href="https://colab.research.google.com/github/fatmasenguler/Mutation_KRAS_analysis/blob/main/5_channels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install biopython networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.8 MB/s eta 0:00:00


In [1]:
from google.colab import files
uploaded = files.upload()


Saving 6GOD.pdb to 6GOD.pdb
Saving 6GOF.pdb to 6GOF.pdb


In [6]:
import numpy as np
import networkx as nx
from Bio.PDB import PDBParser
from numpy.linalg import pinv, det, eigvalsh
import os

# -------------------------------------------------------
#  FUNCTION: Read PDB and build C? graph with distances
# -------------------------------------------------------
def build_ca_graph(pdb_file, cutoff):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", pdb_file)
    model = structure[0]
    chain = list(model.get_chains())[0]

    coords = []
    res_ids = []
    for residue in chain:
        if "CA" in residue:
            coords.append(residue["CA"].coord)
            res_ids.append(residue.get_id()[1])

    coords = np.array(coords)
    n = len(coords)
    G = nx.Graph()
    for i in range(n):
        for j in range(i+1, n):
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist <= cutoff:
                # Store the physical distance dij
                G.add_edge(res_ids[i], res_ids[j], weight=dist)
    return G, res_ids

# -------------------------------------------------------
#  FUNCTION: Global Thermodynamics (F, E, S, Cv)
# -------------------------------------------------------
def compute_global_thermodynamics(G, res_ids, kT):
    n = len(res_ids)
    index = {res_ids[i]: i for i in range(n)}

    def get_log_z(temp):
        L = np.zeros((n, n))
        for u, v, data in G.edges(data=True):
            w = np.exp(-data['weight'] / temp)
            i, j = index[u], index[v]
            L[i, i] += w; L[j, j] += w; L[i, j] -= w; L[j, i] -= w
        L_red = L[:-1, :-1]
        evals = eigvalsh(L_red)
        return np.sum(np.log(evals[evals > 1e-10]))

    h = 1e-4 * kT
    f0 = get_log_z(kT)
    f_plus = get_log_z(kT + h)
    f_minus = get_log_z(kT - h)

    df = (f_plus - f_minus) / (2 * h)
    ddf = (f_plus - 2*f0 + f_minus) / (h**2)

    F = -kT * f0
    E_mean = (kT**2) * df
    Cv = (2 * kT * df) + (kT**2 * ddf)
    S = f0 + (E_mean / kT)

    return F, E_mean, S, Cv

# -------------------------------------------------------
#  FUNCTION: Path Probability (Burton–Pemantle)
# -------------------------------------------------------
def path_probability(path, G, K, index, kT):
    edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
    k = len(edges)
    Ksub = np.zeros((k, k))
    for a, (u1, v1) in enumerate(edges):
        w_a = np.exp(-G[u1][v1]['weight'] / kT)
        for b, (u2, v2) in enumerate(edges):
            i1, j1, i2, j2 = index[u1], index[v1], index[u2], index[v2]
            Ksub[a, b] = w_a * (K[i1, i2] + K[j1, j2] - K[i1, j2] - K[j1, i2])
    return abs(det(Ksub))

# -------------------------------------------------------
#  FUNCTION: Find simple paths (DFS)
# -------------------------------------------------------
def find_paths_of_length(G, start, end, length):
    paths = []
    stack = [(start, [start])]
    while stack:
        node, path = stack.pop()
        if len(path) == length:
            if node == end: paths.append(path)
            continue
        if len(path) < length:
            for neighbor in sorted(G.neighbors(node)):
                if neighbor not in path:
                    stack.append((neighbor, path + [neighbor]))
    return paths

# -------------------------------------------------------
#  MAIN EXECUTION
# -------------------------------------------------------
def main():
    # 1. Inputs
    pdb_file = input("Enter PDB filename: ").strip()
    pdb_base = os.path.splitext(pdb_file)[0]
    cutoff = float(input("Enter Cutoff Distance (e.g. 8.0): "))
    kT = float(input("Enter kT (e.g. 1.0): "))
    start_res = int(input("Enter START residue for channel: "))
    end_res = int(input("Enter END residue for channel: "))

    G, res_ids = build_ca_graph(pdb_file, cutoff)

    # 2. Global Calculations
    print(f"\nCalculating Global Thermodynamics for {pdb_file}...")
    gF, gE, gS, gCv = compute_global_thermodynamics(G, res_ids, kT)

    # 3. Setup for Channel Paths
    n = len(res_ids)
    idx_map = {res_ids[i]: i for i in range(n)}
    L = np.zeros((n, n))
    for u, v, data in G.edges(data=True):
        w = np.exp(-data['weight'] / kT)
        i, j = idx_map[u], idx_map[v]
        L[i, i] += w; L[j, j] += w; L[i, j] -= w; L[j, i] -= w
    K = pinv(L)

    all_raw_probs = []
    path_energies = []
    total_paths_found = 0

    print(f"\nSearching channel paths from {start_res} to {end_res}...")
    for length in range(3, 10):
        paths = find_paths_of_length(G, start_res, end_res, length)
        if not paths: continue

        print(f"  Length {length}: Found {len(paths)} paths.")
        total_paths_found += len(paths)

        for p in paths:
            prob = path_probability(p, G, K, idx_map, kT)
            # Physical energy of path = sum of dij distances
            energy = sum(G[p[i]][p[i+1]]['weight'] for i in range(len(p)-1))

            all_raw_probs.append(prob)
            path_energies.append(energy)

    # 4. Channel Thermodynamics Calculation
    if total_paths_found > 0:
        Z_c = sum(all_raw_probs)
        cF = -kT * np.log(Z_c)
        cE = sum(p * e for p, e in zip(all_raw_probs, path_energies)) / Z_c
        cS = (cE - cF) / kT
        E_sq_avg = sum(p * (e**2) for p, e in zip(all_raw_probs, path_energies)) / Z_c
        cCv = (E_sq_avg - cE**2) / (kT**2)
    else:
        print("No paths found for the given channel residues.")
        return

    # 5. Output Results
    output_name = f"{pdb_base}_channel_{start_res}_{end_res}.txt"
    with open(output_name, "w") as f:
        f.write(f"THERMODYNAMIC SUMMARY: {pdb_file}\n")
        f.write(f"Parameters: kT={kT}, Cutoff={cutoff}\n")
        f.write("-" * 50 + "\n")
        f.write(f"GLOBAL PROTEIN DATA:\n")
        f.write(f"  F: {gF:.4f} | E: {gE:.4f} | S: {gS:.4f} | Cv: {gCv:.4f}\n\n")
        f.write(f"CHANNEL DATA (Res {start_res} -> {end_res}):\n")
        f.write(f"  Total paths searched (Len 3-9): {total_paths_found}\n")
        f.write(f"  Channel Free Energy (Fc):  {cF:.4f}\n")
        f.write(f"  Channel Mean Energy (Ec):  {cE:.4f}\n")
        f.write(f"  Channel Entropy (Sc):      {cS:.4f}\n")
        f.write(f"  Channel Heat Capacity (Cvc): {cCv:.4f}\n")

    print(f"\nCalculations complete. Summary saved to: {output_name}")

if __name__ == "__main__":
    main()

Enter PDB filename: 6GOF.pdb
Enter Cutoff Distance (e.g. 8.0): 8.0
Enter kT (e.g. 1.0): 1.0
Enter START residue for channel: 12
Enter END residue for channel: 61

Calculating Global Thermodynamics for 6GOF.pdb...

Searching channel paths from 12 to 61...
  Length 3: Found 3 paths.
  Length 4: Found 17 paths.
  Length 5: Found 92 paths.
  Length 6: Found 551 paths.
  Length 7: Found 3785 paths.
  Length 8: Found 28222 paths.
  Length 9: Found 217938 paths.

Calculations complete. Summary saved to: 6GOF_channel_12_61.txt
